# Extract.ipynb
This file will preprocess and extract neccessary info from `camma_mvor_2018.json` annotation file 

In [ ]:
#importing neccessary packages and libraries
import json
from collections import defaultdict
import numpy as np

In [74]:
#loading in pose annotations 
try:
    with open('camma_mvor_2018.json') as f:
        data = json.load(f)
except FileNotFoundError:
    print("file not found")

except:
    print("other error")

annot_3D = data["annotations3D"] #3D pose annotations


#ensuring annotations where loaded properly
print(annot_3D[0]['keypoints3D'])

[630.6556928399972, -498.0674644841253, 2394.9087204313873, 2.0, 650.6029327608261, -435.673747489747, 2467.1504194692443, 2.0, 765.2336487731467, -415.1274711986172, 2567.2710725550846, 2.0, 471.51789534403446, -417.4315698597386, 2576.9883790848066, 2.0, 762.8635121989911, -103.45690191919803, 2816.927522526169, 2.0, 493.88975427043096, -88.01155362794417, 2806.6194635029433, 2.0, 739.0484619140625, -224.3656005859375, 2560.0, 2.0, 334.1165332364915, -217.05363208174617, 2561.7882492948274, 2.0, 737.6575317382812, -122.52092742919922, 2450.0, 2.0, 478.4024783369531, -167.64844196983597, 2436.253111187903, 2.0, 787.6575317382812, -72.52092742919922, 2500.0, 1.0, 541.8294915212192, -182.77253333431509, 2388.574626849282, 2.0]


## Validating keypoint shape

Looking at the first annotation entry, we see notice that first keypoint entry has more than 10 keypoints.


In [75]:
print(len(annot_3D[0]['keypoints3D']))

48


Given that each keypoint is of the form x,y,z,?  if there were 10 keypoints there should be the lenght of the entry should be 4x10=40. To verify if this is consistent across the whole dataset we will check keypoint count for every entry

In [76]:
lengths = [len(a['keypoints3D']) for a in annot_3D]
print(set(lengths))  # if one number, it's consistent

{48}


The code above proves a disrepency between, documentation and actual data shape. After further inspection, 4th coordinate seems to be irrelevant so it will be dropped from every frame. As for the the 11 and 12 keypoints respectively, after visualization they seem to represent keypoints for the hands, thus they will be left in.

## Reshaping annotations

48 value keypoint will now become a 12x3 list (12 keypoints , xyz coordinates)

In [77]:
def convertKeypoints(keypoints):
    coords = []

    #removing mystery coordinate
    for i, coord in enumerate(keypoints):
        if i % 4 != 3:
            coords.append(coord)

    # reshape to 12x3
    joints = []
    for i in range(0, len(coords), 3):
        joints.append(coords[i:i+3])

    return joints

def printKp(keypoints): #helper to print keypoints joint by joint
    for i in range(len(keypoints)):
        print(f"{i+1}: {keypoints[i]}")

#quick test 
kp = [630.6556928399972, -498.0674644841253, 2394.9087204313873, 2.0, 650.6029327608261, -435.673747489747, 2467.1504194692443, 2.0, 765.2336487731467, -415.1274711986172, 2567.2710725550846, 2.0, 471.51789534403446, -417.4315698597386, 2576.9883790848066, 2.0, 762.8635121989911, -103.45690191919803, 2816.927522526169, 2.0, 493.88975427043096, -88.01155362794417, 2806.6194635029433, 2.0, 739.0484619140625, -224.3656005859375, 2560.0, 2.0, 334.1165332364915, -217.05363208174617, 2561.7882492948274, 2.0, 737.6575317382812, -122.52092742919922, 2450.0, 2.0, 478.4024783369531, -167.64844196983597, 2436.253111187903, 2.0, 787.6575317382812, -72.52092742919922, 2500.0, 1.0, 541.8294915212192, -182.77253333431509, 2388.574626849282, 2.0]
printKp(convertKeypoints(kp))



1: [630.6556928399972, -498.0674644841253, 2394.9087204313873]
2: [650.6029327608261, -435.673747489747, 2467.1504194692443]
3: [765.2336487731467, -415.1274711986172, 2567.2710725550846]
4: [471.51789534403446, -417.4315698597386, 2576.9883790848066]
5: [762.8635121989911, -103.45690191919803, 2816.927522526169]
6: [493.88975427043096, -88.01155362794417, 2806.6194635029433]
7: [739.0484619140625, -224.3656005859375, 2560.0]
8: [334.1165332364915, -217.05363208174617, 2561.7882492948274]
9: [737.6575317382812, -122.52092742919922, 2450.0]
10: [478.4024783369531, -167.64844196983597, 2436.253111187903]
11: [787.6575317382812, -72.52092742919922, 2500.0]
12: [541.8294915212192, -182.77253333431509, 2388.574626849282]


In [78]:
#reshaping all keypoints
for annot in annot_3D:
    annot['keypoints3D']=convertKeypoints(annot['keypoints3D'])




In [79]:
printKp(annot_3D[0]['keypoints3D'])

1: [630.6556928399972, -498.0674644841253, 2394.9087204313873]
2: [650.6029327608261, -435.673747489747, 2467.1504194692443]
3: [765.2336487731467, -415.1274711986172, 2567.2710725550846]
4: [471.51789534403446, -417.4315698597386, 2576.9883790848066]
5: [762.8635121989911, -103.45690191919803, 2816.927522526169]
6: [493.88975427043096, -88.01155362794417, 2806.6194635029433]
7: [739.0484619140625, -224.3656005859375, 2560.0]
8: [334.1165332364915, -217.05363208174617, 2561.7882492948274]
9: [737.6575317382812, -122.52092742919922, 2450.0]
10: [478.4024783369531, -167.64844196983597, 2436.253111187903]
11: [787.6575317382812, -72.52092742919922, 2500.0]
12: [541.8294915212192, -182.77253333431509, 2388.574626849282]


## Frame by frame pose grouping 
Relevant information to extraction from each annotation
- `image_ids`
- `person_id`
- `keypoints3D`

Poses per frame will be stored in dictionary with the following shape: 

```python
    poses_per_frame = {
        frame_1: [ {pose1}, ... , {poseN}],
        ...
        frame_N: [ {pose1}, ... , {poseN}],
    }
```



In [80]:
poses_per_frame = defaultdict(list) 

for annot in annot_3D:
    frame = annot['image_ids']
    person = annot['person_id']
    keypoints = annot['keypoints3D']


    poses_per_frame[frame].append({
        'person': person,
        'keypoints': keypoints
    })

#inspect data to make sure it was parsed properly 
print(f"There's {len(poses_per_frame)} total frames") 
print(f"Frame 1: {list(poses_per_frame.values())[0]}")

multiPersonFrame = 0
for pose in list(poses_per_frame.values()):
    if len(pose)>1:
        multiPersonFrame +=1

print(f"Theres {multiPersonFrame} frames with more than one person")


There's 629 total frames
Frame 1: [{'person': 0, 'keypoints': [[630.6556928399972, -498.0674644841253, 2394.9087204313873], [650.6029327608261, -435.673747489747, 2467.1504194692443], [765.2336487731467, -415.1274711986172, 2567.2710725550846], [471.51789534403446, -417.4315698597386, 2576.9883790848066], [762.8635121989911, -103.45690191919803, 2816.927522526169], [493.88975427043096, -88.01155362794417, 2806.6194635029433], [739.0484619140625, -224.3656005859375, 2560.0], [334.1165332364915, -217.05363208174617, 2561.7882492948274], [737.6575317382812, -122.52092742919922, 2450.0], [478.4024783369531, -167.64844196983597, 2436.253111187903], [787.6575317382812, -72.52092742919922, 2500.0], [541.8294915212192, -182.77253333431509, 2388.574626849282]]}]
Theres 313 frames with more than one person


## Per operation grouping
Based on pose grouping above, frames will now be grouped per operation. Operations will be stored in a dictionary with the following shape:

```python
    operations = {
        day_x: {operation_x: [frames]}

    }
```

In [ ]:
OP_RANGES = { #frame span of operations
    'day1': {
        (0, 56)
    },
    'day2': {
        (0, 117),
        (118, 232),
        (233, 234),
        (235, 245),
        (246, 257),
        (258, 270),
        (271, 278),
        (279, 293),
        (294, 321),
        (322, 324),
        (325, 329)
    },
    'day3': {
        (0, 7),
        (8, 28),
        (29, 58),
        (59, 84),
        (85, 124),
        (125, 138),
        (139, 160),
        (161, 172),
        (173, 181),
        (182, 196),
        (197, 217)
    },
    'day4': {
        (0, 75),
        (76, 116),
        (117, 121)
    }
}

operations = defaultdict(lambda: defaultdict(list)) 
def getIndex(imageId): #extracts image index from id 
    return int(imageId.split('_')[0][-6:]) #ex: 10010000002_10020000002_10030000002 --> 000002 --> 2)

def getDay(imageId): #extracts day from id
    day = int(imageId.split('_')[0][0])

    match day:
        case 1:
            return 'day1'
        
        case 2:
            return 'day2'
        
        case 3:
            return 'day3'
        
        case 4:
            return 'day4'
    

#iterating through frames and mapping them to their respective operations 
for frameId, poses in poses_per_frame.items():
    dayString = getDay(frameId)
    frameIdx = getIndex(frameId)


    for opIdx, (opStart, opEnd) in enumerate(OP_RANGES[dayString], start =1):
        
        if opStart <= frameIdx <= opEnd:
            opString = f"operation_{opIdx}"
            operations[dayString][opString].append({
                'frameIdx': frameIdx,
                'poses': poses 
            })
            break #make sure it only maps to one operation

#sort to put frames in order
for day in operations:
    for op in operations[day]:
        operations[day][op].sort(key=lambda f: f['frameIdx'])



### Inspecting data 

In [82]:
#Visualizing operation annotations
print(f"Day 1, operation 1: {operations['day1']['operation_1']}")

Day 1, operation 1: [{'frameIdx': 1, 'poses': [{'person': 0, 'keypoints': [[-919.0739482366955, 220.82523959568664, 1500.285433128346], [-900.6188055769591, 269.0215963454077, 1591.8862113213054], [-767.3401648708726, 254.80329475774147, 1701.0095984835384], [-1062.8579020560446, 242.84780550162645, 1779.6725563248588], [-707.1937682188919, 473.3078621762493, 1983.514584502264], [-971.708986329872, 629.1873350132807, 1986.7901324341426], [-642.777364167334, 357.5466914290271, 1908.2174367030243], [-1093.1801297857166, 456.7583338835688, 1950.6948686629373], [-602.3968193964831, 574.8271083699042, 1919.106531333714], [-1117.1871539088602, 665.6120719028361, 2062.1952132462443], [-596.7772216796875, 643.4946899414062, 1883.0], [-1154.817825383358, 691.7864954734459, 2068.0941845246466]]}]}, {'frameIdx': 3, 'poses': [{'person': 1, 'keypoints': [[-227.58989221610074, -184.0425039183756, 2313.0511787435516], [-233.268458343472, -170.6410320503951, 2410.820223440352], [-68.43472747519547, -6

In [85]:
#Seeing how many operations have under 10 annotated frames 

under10 = 0
total = 0

for day in operations:
    for op in operations[day]:
        total+=1
        if len(operations[day][op])<10:
            under10+=1

print(f"{under10}/{total} operations have less than 10 frames")
    

9/26 operations have less than 10 frames


## Building frame vector

shape: `[Mean person config  | frame diversity | Scene  graph summary ]`

In [ ]:
## exporting per operation grouping to work with cleaner data

with open('poses_per_operation.json', 'w') as f:
    json.dump(operations, f)

In [94]:
try:
    with open('poses_per_operation.json') as f:
        cleanedAnnots = json.load(f) #cleaned annotations
except FileNotFoundError:
    print("file not found")

except:
    print("other error")

In [98]:
## helpers to print operations in a cleaner manner (for visuals)
exFrame = {"frameIdx": 125, "poses": [{"person": 0, "keypoints": [[425.9325329990054, 3.702993114246245, 1715.4962845313348], [394.4498771775684, 84.61828843598573, 1740.7303525692053], [279.6814298393938, 111.09525093472891, 1861.2820083885113], [518.4143084994325, 231.81319148895287, 1739.4548896488695], [281.28080513341644, 452.0650229333778, 2085.401214097045], [500.97187757653046, 537.1034976491628, 1971.4877231176597], [238.14033883746475, 284.5853595779416, 2098.9022304218324], [522.8756265637255, 470.29348214448964, 1867.9026586227037], [250.89139830689237, 370.24345778978693, 2189.8112687419784], [691.8487795971357, 344.8215830036688, 1839.519465041534], [279.3292376159384, 410.517108668515, 2234.367598574191], [754.1095502434551, 332.34481114871846, 1821.852929467502]]}]}

def printFrame(frame):
    print(f"Frame: {frame['frameIdx']}")
    print("---------Poses----------")
    for pose in frame["poses"]:
        print(f"Person {pose["person"]}")
        print("Keypoints:")
        printKp(pose["keypoints"])
        print("\n")
    

In [99]:
#testing 
printFrame(exFrame)

Frame: 125
---------Poses----------
Person 0
Keypoints:
1: [425.9325329990054, 3.702993114246245, 1715.4962845313348]
2: [394.4498771775684, 84.61828843598573, 1740.7303525692053]
3: [279.6814298393938, 111.09525093472891, 1861.2820083885113]
4: [518.4143084994325, 231.81319148895287, 1739.4548896488695]
5: [281.28080513341644, 452.0650229333778, 2085.401214097045]
6: [500.97187757653046, 537.1034976491628, 1971.4877231176597]
7: [238.14033883746475, 284.5853595779416, 2098.9022304218324]
8: [522.8756265637255, 470.29348214448964, 1867.9026586227037]
9: [250.89139830689237, 370.24345778978693, 2189.8112687419784]
10: [691.8487795971357, 344.8215830036688, 1839.519465041534]
11: [279.3292376159384, 410.517108668515, 2234.367598574191]
12: [754.1095502434551, 332.34481114871846, 1821.852929467502]




In [95]:
def printOp(operation):
    for frame in operation:
        printFrame(frame)

In [96]:
#testing 
printOp(cleanedAnnots["day1"]["operation_1"])

Frame: 1
---------Poses----------
Person 0
Keypoints
1: [-919.0739482366955, 220.82523959568664, 1500.285433128346]
2: [-900.6188055769591, 269.0215963454077, 1591.8862113213054]
3: [-767.3401648708726, 254.80329475774147, 1701.0095984835384]
4: [-1062.8579020560446, 242.84780550162645, 1779.6725563248588]
5: [-707.1937682188919, 473.3078621762493, 1983.514584502264]
6: [-971.708986329872, 629.1873350132807, 1986.7901324341426]
7: [-642.777364167334, 357.5466914290271, 1908.2174367030243]
8: [-1093.1801297857166, 456.7583338835688, 1950.6948686629373]
9: [-602.3968193964831, 574.8271083699042, 1919.106531333714]
10: [-1117.1871539088602, 665.6120719028361, 2062.1952132462443]
11: [-596.7772216796875, 643.4946899414062, 1883.0]
12: [-1154.817825383358, 691.7864954734459, 2068.0941845246466]


Frame: 3
---------Poses----------
Person 1
Keypoints
1: [-227.58989221610074, -184.0425039183756, 2313.0511787435516]
2: [-233.268458343472, -170.6410320503951, 2410.820223440352]
3: [-68.434727475

### GPD descriptor

defining functions and constants needed to build gpd feature vector 

In [100]:
#mapping joints to indices in keypoints 
JOINTS = {'head': 0, 'neck': 1, 'left_shoulder': 2, 'right_shoulder': 3, 'left_hip': 4, 'right_hip': 5, 'left_elbow': 6, 'right_elbow': 7, 'left_wrist': 8, 'right_wrist': 9, 'left_hand': 10, 'right_hand': 11}

#defining planes

PLANES = ['torso', (2,6,10), (3,7,11)] #torso needed to be dervied 

LINES = [
    #12 direcrtly adjacent lines
    (0, 1),   # head - neck
    (1, 2),   # neck - left_shoulder
    (1, 3),   # neck - right_shoulder
    (2, 4),   # left_shoulder - left_hip
    (3, 5),   # right_shoulder - right_hip
    (4, 5),   # left_hip - right_hip
    (2, 6),   # left_shoulder - left_elbow
    (3, 7),   # right_shoulder - right_elbow
    (6, 8),   # left_elbow - left_wrist
    (7, 9),   # right_elbow - right_wrist
    (8, 10),  # left_wrist - left_hand
    (9, 11),  # right_wrist - right_hand

    #Type 2 lines
    (0, 2),   # head - left_shoulder
    (0, 3),   # head - right_shoulder
    (10, 6),  # left_hand - left_shoulder
    (11, 7),  # right_hand - right_shoulder

    #Type 3 lines
    (0, 10),  # head - left_hand
    (0, 11),  # head - right_hand
    (10, 11), # left_hand - right_hand
]

In [109]:
## converting keypoint arrays into numpy arrays
for day in cleanedAnnots:
    for operation in cleanedAnnots[day]:
        for frame in cleanedAnnots[day][operation]:
            for pose in frame['poses']:
                pose['keypoints'] = np.array(pose['keypoints'])



In [110]:
##testing validity of keypoint structure
all_valid = True

for day in cleanedAnnots:
    for operation in cleanedAnnots[day]:
        for frame in cleanedAnnots[day][operation]:
            for pose in frame['poses']:
                kp = pose['keypoints']
                if not isinstance(kp, np.ndarray):
                    print(f"Not numpy array: day={day}, op={operation}, frame={frame['frameIdx']}, person={pose['person']}")
                    all_valid = False
                elif kp.shape != (12, 3):
                    print(f"Wrong shape {kp.shape}: day={day}, op={operation}, frame={frame['frameIdx']}, person={pose['person']}")
                    all_valid = False

if all_valid:
    print("All keypoints valid: numpy arrays of shape (12, 3)")

All keypoints valid: numpy arrays of shape (12, 3)


In [ ]:
##exporting cleaned annotations so it can be worked on in gpd notebook